# 04 — Uplift Model Training: T-Learner, Class Transformation & Spend Uplift

## Objective
Train the core heterogeneous treatment-effect models.

Models:
1. Two-Model / T-Learner — Men's Email vs Control
2. Class Transformation — Men's Email vs Control
3. Two-Model / T-Learner — Women's Email vs Control
4. Class Transformation — Women's Email vs Control
5. Two-Model / T-Learner — Any Email vs Control
6. Class Transformation — Any Email vs Control
7. Optional spend-uplift regressors

Predictions are generated on the held-out test set and on all customers for downstream scoring.

In [ ]:
from pathlib import Path
import sys, json, warnings
warnings.filterwarnings("ignore")

# Find the project root so the notebook works whether launched from the project root
# or from inside the notebooks directory.
_here = Path.cwd().resolve()
_candidates = [_here, *_here.parents]
ROOT = next((p for p in _candidates if (p / "src" / "config.py").exists()), _here)
if not (ROOT / "src" / "config.py").exists():
    raise FileNotFoundError("Could not locate project root containing src/config.py")

sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)

In [ ]:
import numpy as np
import pandas as pd
import joblib
from src.data_utils import load_data, prepare
from src.feature_engineering import make_preprocessor
from src.uplift_models import train_tlearner, predict_tlearner, train_class_transformation, save_model
from src.config import DATA_PATH, RANDOM_SEED, TEST_SIZE, MODELS_DIR, PREDICTIONS_DIR, MODEL_PARAMS
from sklearn.model_selection import train_test_split

df = prepare(load_data(DATA_PATH))
idx_train, idx_test = train_test_split(
    np.arange(len(df)), test_size=TEST_SIZE,
    random_state=RANDOM_SEED, stratify=df["segment"]
)
train = df.iloc[idx_train].reset_index(drop=True)
test = df.iloc[idx_test].reset_index(drop=True)

pre = make_preprocessor()
X_train = pre.fit_transform(train)
X_test = pre.transform(test)
X_all = pre.transform(df)

y_train = train["conversion"].to_numpy()
y_test = test["conversion"].to_numpy()
t_train = train["segment"].to_numpy()
t_test = test["segment"].to_numpy()

print("Train:", X_train.shape, "Test:", X_test.shape)

## 1. Men's Email vs Control

In [ ]:
models = {}
pred_test = pd.DataFrame({
    "segment": test.segment.to_numpy(),
    "conversion": y_test
})
pred_all = pd.DataFrame({
    "segment": df.segment.to_numpy(),
    "conversion": df.conversion.to_numpy()
})

for arm, slug in [("Mens E-Mail", "mens"), ("Womens E-Mail", "womens")]:
    mt, mc = train_tlearner(X_train, t_train, y_train, arm)
    pt, pc = predict_tlearner(mt, mc, X_test)
    ft, fc = predict_tlearner(mt, mc, X_all)

    ct, propensity = train_class_transformation(X_train, t_train, y_train, arm)
    uplift_ct_test = ct.predict(X_test)
    uplift_ct_all = ct.predict(X_all)

    models[f"tlearner_{slug}_treatment"] = mt
    models[f"tlearner_{slug}_control"] = mc
    models[f"class_transform_{slug}"] = (ct, propensity)

    pred_test[f"pred_conversion_{slug}"] = pt
    pred_test[f"pred_conversion_control_{slug}"] = pc
    pred_test[f"uplift_{slug}"] = pt - pc
    pred_test[f"uplift_{slug}_class_transform"] = uplift_ct_test

    pred_all[f"pred_conversion_{slug}"] = ft
    pred_all[f"pred_conversion_control_{slug}"] = fc
    pred_all[f"uplift_{slug}"] = ft - fc
    pred_all[f"uplift_{slug}_class_transform"] = uplift_ct_all

    save_model(mt, f"notebook_04_tlearner_{slug}_treatment")
    save_model(mc, f"notebook_04_tlearner_{slug}_control")
    save_model((ct, propensity), f"notebook_04_class_transform_{slug}")

print("Men's and Women's models trained.")

## 2. Any Email vs Control

In [ ]:
t_train_any = np.where(t_train == "No E-Mail", "No E-Mail", "Any Email")
t_test_any = np.where(t_test == "No E-Mail", "No E-Mail", "Any Email")

mt_any, mc_any = train_tlearner(X_train, t_train_any, y_train, "Any Email")
pt_any, pc_any = predict_tlearner(mt_any, mc_any, X_test)
ft_any, fc_any = predict_tlearner(mt_any, mc_any, X_all)

ct_any, propensity_any = train_class_transformation(X_train, t_train_any, y_train, "Any Email")

pred_test["pred_conversion_any"] = pt_any
pred_test["pred_conversion_control_any"] = pc_any
pred_test["uplift_any_email"] = pt_any - pc_any
pred_test["uplift_any_email_class_transform"] = ct_any.predict(X_test)

pred_all["pred_conversion_any"] = ft_any
pred_all["pred_conversion_control_any"] = fc_any
pred_all["uplift_any_email"] = ft_any - fc_any
pred_all["uplift_any_email_class_transform"] = ct_any.predict(X_all)

save_model(mt_any, "notebook_04_tlearner_any_treatment")
save_model(mc_any, "notebook_04_tlearner_any_control")
save_model((ct_any, propensity_any), "notebook_04_class_transform_any")

display(pred_test.head())

## 3. Optional spend-uplift models

In [ ]:
from lightgbm import LGBMRegressor

spend_models = {}
for arm, slug in [("Mens E-Mail","mens"), ("Womens E-Mail","womens")]:
    mask_t = train.segment.to_numpy() == arm
    mask_c = train.segment.to_numpy() == "No E-Mail"
    rm_t = LGBMRegressor(**dict(MODEL_PARAMS, random_state=RANDOM_SEED))
    rm_c = LGBMRegressor(**dict(MODEL_PARAMS, random_state=RANDOM_SEED))
    rm_t.fit(X_train[mask_t], train.loc[mask_t, "spend"])
    rm_c.fit(X_train[mask_c], train.loc[mask_c, "spend"])
    pred_test[f"spend_uplift_{slug}"] = rm_t.predict(X_test) - rm_c.predict(X_test)
    pred_all[f"spend_uplift_{slug}"] = rm_t.predict(X_all) - rm_c.predict(X_all)
    spend_models[slug] = (rm_t, rm_c)
    save_model(rm_t, f"notebook_04_spend_{slug}_treatment")
    save_model(rm_c, f"notebook_04_spend_{slug}_control")

# Any Email spend uplift
mask_t = t_train_any == "Any Email"
mask_c = t_train_any == "No E-Mail"
rm_t = LGBMRegressor(**dict(MODEL_PARAMS, random_state=RANDOM_SEED))
rm_c = LGBMRegressor(**dict(MODEL_PARAMS, random_state=RANDOM_SEED))
rm_t.fit(X_train[mask_t], train.loc[mask_t, "spend"])
rm_c.fit(X_train[mask_c], train.loc[mask_c, "spend"])
pred_test["spend_uplift_any"] = rm_t.predict(X_test) - rm_c.predict(X_test)
pred_all["spend_uplift_any"] = rm_t.predict(X_all) - rm_c.predict(X_all)

save_model(rm_t, "notebook_04_spend_any_treatment")
save_model(rm_c, "notebook_04_spend_any_control")

print("Spend uplift models trained.")

In [ ]:
PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)
pred_test.to_csv(PREDICTIONS_DIR / "notebook_04_test_predictions.csv", index=False)
pred_all.to_csv(PREDICTIONS_DIR / "notebook_04_all_predictions.csv", index=False)
print("Saved test and all-customer predictions.")

## Interpretation
The T-Learner estimates two conditional response surfaces and subtracts them.

The class-transformation model directly learns a transformed treatment effect target. Neither approach observes an individual's true counterfactual outcome; both produce **model-estimated heterogeneous treatment effects**.